# Erste Datenaufbereitung
## Ueberwachungskameras
Datum: 27.01.2022

Dieses Jupyter Notebook hilft, die Daten einfacher für den Datenkatalog verfügbar zu machen.

1. Befragungsdaten einlesen
2. Aus den Variabelnamen und Labels ein Excel exportieren, welches für CopyPaste ins Metadatenexcelfile verwendet werden kann
3. Variablen und deren Werte aus zwei Excelsheets zu einer Codeliste in csv generieren und exportieren 

Mit Colab kann **dieses Jupyter-Notebook interaktiv im Browser** gestartet werden:
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)]()



**Importiere die notwendigen Packages**

In [1]:
#%pip install geopandas altair fiona requests folium mplleaflet contextily seaborn datetime plotly

In [2]:
import pandas as pd
import datetime
import time
import numpy as np


import pivottablejs
from pivottablejs import pivot_ui
import numpy as np
import altair as alt
import matplotlib.pyplot as plt
import geopandas as gpd
import folium 
import requests

In [3]:
import plotly.express as px


Definiere Settings. Hier das Zahlenformat von Float-Werten (z.B. *'{:,.2f}'.format* mit Komma als Tausenderzeichen), 

In [25]:
pd.options.display.float_format = '{:.5f}'.format
pd.set_option('display.width', 100)
pd.set_option('display.max_columns', 150)

## Daten importieren

**Dataset auf INTEG Datenkatalog**: 
**Dataset auf PROD Datenkatalog**: 
 

In [26]:
#Die Datasets sind nur zum Testen auf INT-DWH-Dropzone. Wenn der Test vorbei ist, sind sie auf PROD. 
# Über den Status kann man einfach switchen

status = "test";
print(status)

test


### Pfade setzen

In [27]:
# Filepath
if status == "prod":
    print("ne, nix prod")
else:
    fp_data = r"\\szh\ssz\applikationen\OGD\Daten\Quelldaten\STEZ\ueberwachungskameras\dataprep\20220114_Überwachungskameras_OpenData_OGD.xlsx"
    fp_output = r"\\szh\ssz\applikationen\OGD\Daten\Quelldaten\STEZ\ueberwachungskameras\dataprep"
     


### Importiere den Testdatensatz

Importiere 2021er-Variabeln

In [28]:
imp_ueberwachungscams = pd.pandas.read_excel(
    fp_data 
    , sheet_name=0
    , header=0
    , index_col=None
    , usecols=None, skiprows=None, nrows=None, na_values=None, keep_default_na=True, na_filter=True, parse_dates=False, date_parser=None, thousands=None
    , comment=None, skipfooter=0, convert_float=None, mangle_dupe_cols=True, storage_options=None
    )

In [29]:
imp_ueberwachungscams.head(2)

,Name Standort,Adresse,Breitengrad,Längengrad,Anzahl Videokameras Aussenbereich,Anzahl Videokameras Innenbereich,Anzahl Videokameras Gegensprechanlage,Bereiche der Überwachung,Aufbewahrungsdauer (Tage),Verantwortliche Dienstabteilung,Rechtsgrundlage
0,EWZ Unterwerk Auwiesen,"Aubruggweg 21, 8050 Zürich, Switzerland",47.41234,8.57087,1.00000,0.00000,0.00000,Einfahrtstor zum Haupteingang des UW Auwiesen ...,15,Elektrizitätswerk der Stadt Zürich,https://www.stadt-zuerich.ch/portal/de/index/p...
1,EWZ Werkhof Herdern,"Pfingstweidstrasse 85, 8005 Zürich, Switzerland",47.38923,8.51231,1.00000,0.00000,0.00000,Hauptschiebetor bei Haupteinfahrt,15,Elektrizitätswerk der Stadt Zürich,https://www.stadt-zuerich.ch/portal/de/index/p...


In [30]:
imp_ueberwachungscams.columns

Index(['Name Standort', 'Adresse', 'Breitengrad', 'Längengrad',
       'Anzahl Videokameras Aussenbereich', 'Anzahl Videokameras Innenbereich',
       'Anzahl Videokameras Gegensprechanlage', 'Bereiche der Überwachung',
       'Aufbewahrungsdauer (Tage)', 'Verantwortliche Dienstabteilung', 'Rechtsgrundlage'],
      dtype='object')

### Rename die Attribute

Alles klein und zusammenhängend beschrieben.

In [31]:
rename_cols = {'Name Standort' : 'standort_beschreibung' 
               ,'Adresse' : 'adresse_beschreibung'
               , 'Breitengrad' : 'lat'
               , 'Längengrad' : 'lon'
               , 'Anzahl Videokameras Aussenbereich' : 'anzahl_kameras_aussen'
               , 'Anzahl Videokameras Innenbereich' : 'anzahl_kameras_innen'
               , 'Anzahl Videokameras Gegensprechanlage' : 'anzahl_kameras_gsa'
               , 'Bereiche der Überwachung' : 'bereich_detail_beschreibung'
               , 'Aufbewahrungsdauer (Tage)' : 'aufbewahrungsdauer'
               , 'Verantwortliche Dienstabteilung' : 'verantwortliche_da'
               , 'Rechtsgrundlage' : 'rechtsgrundlage_url'
}

columns_to_drop ={}

In [32]:
df_ueberwachungscams = imp_ueberwachungscams \
    .copy() \
    .drop(columns_to_drop, axis=1) \
    .rename(columns=rename_cols) \
    .assign(
        #spots_plz=lambda x: x.groupby(['district', 'plz'])['spots'].transform(np.sum)
    )

In [39]:
df_ueberwachungscams.dtypes
df_ueberwachungscams

,standort_beschreibung,adresse_beschreibung,lat,lon,anzahl_kameras_aussen,anzahl_kameras_innen,anzahl_kameras_gsa,bereich_detail_beschreibung,aufbewahrungsdauer,verantwortliche_da,rechtsgrundlage_url
0,EWZ Unterwerk Auwiesen,"Aubruggweg 21, 8050 Zürich, Switzerland",47.41234,8.57087,1.00000,0.00000,0.00000,Einfahrtstor zum Haupteingang des UW Auwiesen ...,15,Elektrizitätswerk der Stadt Zürich,https://www.stadt-zuerich.ch/portal/de/index/p...
1,EWZ Werkhof Herdern,"Pfingstweidstrasse 85, 8005 Zürich, Switzerland",47.38923,8.51231,1.00000,0.00000,0.00000,Hauptschiebetor bei Haupteinfahrt,15,Elektrizitätswerk der Stadt Zürich,https://www.stadt-zuerich.ch/portal/de/index/p...
2,Museum Rietberg,"Gablerstrasse 17, 8002 Zürich, Switzerland",47.35889,8.52974,0.00000,29.00000,0.00000,Aussen-/ Innenbereich,14,Museum Rietberg,https://www.stadt-zuerich.ch/portal/de/index/p...
3,"Organisation- und Informatik Stadt Zürich, Geb...","Albisriederstrasse 201, 8047 Zürich, Switzerland",47.37748,8.49547,1.00000,0.00000,0.00000,Eingangsschleuse,17,Organisation und Information Zürich,https://www.stadt-zuerich.ch/portal/de/index/p...
4,"Stadtspital Zürich, Standort Triemli","Birmensdorferstrasse 497, 8055 Zürich, Switzer...",47.36580,8.49738,274.00000,25.00000,0.00000,"Ein-/Ausgänge für Personen, Ein-/Ausfahrten fü...",14,Stadtspital Triemli,https://www.stadt-zuerich.ch/portal/de/index/p...
...,...,...,...,...,...,...,...,...,...,...,...
129,Pavillon,"Morgartenstrasse 40, 8004 Zürich, Switzerland",47.37181,8.52652,0.00000,4.00000,0.00000,NaN,20,Immobilien Stadt Zürich,https://www.stadt-zuerich.ch/portal/de/index/p...
130,Bürogebäude (Stadtammann- und Betreibungsamt Z...,"Witikonerstrasse 15, 8032 Zürich, Switzerland",47.36412,8.56706,0.00000,2.00000,0.00000,NaN,20,Immobilien Stadt Zürich,https://www.stadt-zuerich.ch/portal/de/index/p...
131,Wohnhaus,"Neufrankengasse, 8004 Zürich, Switzerland",47.38096,8.52736,2.00000,19.00000,0.00000,NaN,20,Immobilien Stadt Zürich,https://www.stadt-zuerich.ch/portal/de/index/p...
132,Verwaltungszentrum Eggbühl,"Eggbühlstrasse, 8050 Zürich, Switzerland",47.41684,8.54377,0.00000,37.00000,0.00000,NaN,20,Immobilien Stadt Zürich,https://www.stadt-zuerich.ch/portal/de/index/p...


In [43]:
#Im Datensatz kommt Langsteinstrasse vor. Sie heisst aber Langsteinenstrasse
check = df_ueberwachungscams.query('adresse_beschreibung != "Langensteinstrasse 40, 8057 Zürich, Switzerland"')
check


,standort_beschreibung,adresse_beschreibung,lat,lon,anzahl_kameras_aussen,anzahl_kameras_innen,anzahl_kameras_gsa,bereich_detail_beschreibung,aufbewahrungsdauer,verantwortliche_da,rechtsgrundlage_url
0,EWZ Unterwerk Auwiesen,"Aubruggweg 21, 8050 Zürich, Switzerland",47.41234,8.57087,1.00000,0.00000,0.00000,Einfahrtstor zum Haupteingang des UW Auwiesen ...,15,Elektrizitätswerk der Stadt Zürich,https://www.stadt-zuerich.ch/portal/de/index/p...
1,EWZ Werkhof Herdern,"Pfingstweidstrasse 85, 8005 Zürich, Switzerland",47.38923,8.51231,1.00000,0.00000,0.00000,Hauptschiebetor bei Haupteinfahrt,15,Elektrizitätswerk der Stadt Zürich,https://www.stadt-zuerich.ch/portal/de/index/p...
2,Museum Rietberg,"Gablerstrasse 17, 8002 Zürich, Switzerland",47.35889,8.52974,0.00000,29.00000,0.00000,Aussen-/ Innenbereich,14,Museum Rietberg,https://www.stadt-zuerich.ch/portal/de/index/p...
3,"Organisation- und Informatik Stadt Zürich, Geb...","Albisriederstrasse 201, 8047 Zürich, Switzerland",47.37748,8.49547,1.00000,0.00000,0.00000,Eingangsschleuse,17,Organisation und Information Zürich,https://www.stadt-zuerich.ch/portal/de/index/p...
4,"Stadtspital Zürich, Standort Triemli","Birmensdorferstrasse 497, 8055 Zürich, Switzer...",47.36580,8.49738,274.00000,25.00000,0.00000,"Ein-/Ausgänge für Personen, Ein-/Ausfahrten fü...",14,Stadtspital Triemli,https://www.stadt-zuerich.ch/portal/de/index/p...
...,...,...,...,...,...,...,...,...,...,...,...
129,Pavillon,"Morgartenstrasse 40, 8004 Zürich, Switzerland",47.37181,8.52652,0.00000,4.00000,0.00000,NaN,20,Immobilien Stadt Zürich,https://www.stadt-zuerich.ch/portal/de/index/p...
130,Bürogebäude (Stadtammann- und Betreibungsamt Z...,"Witikonerstrasse 15, 8032 Zürich, Switzerland",47.36412,8.56706,0.00000,2.00000,0.00000,NaN,20,Immobilien Stadt Zürich,https://www.stadt-zuerich.ch/portal/de/index/p...
131,Wohnhaus,"Neufrankengasse, 8004 Zürich, Switzerland",47.38096,8.52736,2.00000,19.00000,0.00000,NaN,20,Immobilien Stadt Zürich,https://www.stadt-zuerich.ch/portal/de/index/p...
132,Verwaltungszentrum Eggbühl,"Eggbühlstrasse, 8050 Zürich, Switzerland",47.41684,8.54377,0.00000,37.00000,0.00000,NaN,20,Immobilien Stadt Zürich,https://www.stadt-zuerich.ch/portal/de/index/p...


### Sauberen CSV-Export Überwachungskameradaten

Speichere es so mal als CSV

In [44]:
print(fp_output+r'\df_ueberwachungscams.csv')

\\szh\ssz\applikationen\OGD\Daten\Quelldaten\STEZ\ueberwachungskameras\dataprep\imp_ueberwachungscams.csv


In [45]:
df_ueberwachungscams.to_csv(
    fp_output+r'\df_ueberwachungscams.csv'
    , sep=','
    , encoding='utf-8-sig'
    , quoting=2
    , index=False)

Mache noch einen sauberen csv-export aus python der Befragungsdaten

### Exportiere die Adressen um sie zu georeferenzieren
Für **ArcGIS Online** kann die gesamte csv-Datei hochgeladen und georeferenziert werden. Bei einigen Adressen fehlen noch PLZ und Ort. 

--> Warte noch auf Update von Margot



#### Hier noch ein Test für **Flowmap: https://flowmap.blue/geocoding**

Exportiere nur die Adressen und lass sie in Flowmap geocodieren. Das funktioniert sehr gut, wenn allen Adressen am Ende noch Zürich hinzugefügt wird.
Ein Missmatch gibt es bei Langensteinstrasse die eigentlich Langsteinenstrasse heisst...

In [47]:
# Diese Adressen einlesen in flowmap#
exp_adr4georef = df_ueberwachungscams[['adresse_beschreibung', 'lat', 'lon']]
  

exp_adr4georef .to_csv(
    fp_output+r'\exp_adr4georef.csv'
    , sep=','
    , encoding='utf-8-sig'
    , quoting=2
    , index=False)

Nachdem die Georeferenzierung gemacht ist in Flowmap, diese als CSV abspeichern und hier den Output von Flowmap hier wieder prüfen:

In [48]:
imp_flowmap_blue_georef = pd.pandas.read_csv(
    r'\\szh\ssz\applikationen\OGD\Daten\Quelldaten\STEZ\ueberwachungskameras\dataprep\v01_adressbereinigungen\output_georef_flowmap_blue.csv' 
    , sep=","
    )

In [49]:
imp_flowmap_blue_georef.head(1)

,id,name,lat,lon
0,"""Aubruggweg 21, 8050 Zürich""","Aubruggweg 21, 8050 Zürich, Switzerland",47.41234,8.57087


Diesen Output habe ich bereits in AGOL eingelesen und als Feature Service erstellt. Siehe [**hier**](https://gis-stzh.maps.arcgis.com/home/content.html?view=table&sortOrder=desc&sortField=modified&folder=17ad1c550e0745918d7ef0b037eed0ac#content)

## Testing

 - 1) Zeige eine kurze Vorschau der importierten Daten
 - 2) Weise die Datentypen aus
 - 3) Zeige die Shape (Umfang) des Datensatzes an

In [35]:
imp_ueberwachungscams.shape

(136, 23)

**Selektionen und Aggregationen**

Check this [Doku](https://www.shanelynn.ie/summarising-aggregation-and-grouping-data-in-python-pandas/)

Welche aggregierten Todesursachen gibt es überhaupt?

In [82]:
imp_ueberwachungscams.dtypes

Typ                                      object
Standort                                 object
Adresse                                  object
IMMO-Nr. (falls zutreffend)              object
Lat                                     float64
Long                                    float64
Aussen analog                           float64
Aussen digital                          float64
Innen analog                            float64
Innen digital                           float64
Portrait (Ein-bau in GSA)               float64
Undefiniert                             float64
Gebäude-zugang                           object
Schalter-bereich                         object
Liftvor-platz                            object
Fassade                                  object
Eingangsbereich                          object
Bereiche der Überwachung                 object
Zusätzliche Bereiche der Überwachung     object
Aufbewahrungsdauer (Tage)                 int64
Verantwortliche Dienstabteilung         

In [47]:
cams_per_da = imp_ueberwachungscams.groupby(['Verantwortliche Dienstabteilung'])[['Aussen digital','Innen digital','Portrait (Ein-bau in GSA)','Undefiniert']].sum()
cams_per_da.reset_index().sort_values('Verantwortliche Dienstabteilung', ascending=True)
#cams_per_da_aufbewahrung.reset_index().melt()

,Verantwortliche Dienstabteilung,Aussen digital,Innen digital,Portrait (Ein-bau in GSA),Undefiniert
0,EWZ,1,0,0,0
1,IMMO,857,105,27,0
2,LSZ,0,33,0,0
3,MRZ,0,29,0,0
4,OIZ,1,0,0,0
5,STAPO,0,0,0,0
6,Stadtspital Triemli,274,25,0,0
7,Stadtspital Waid,11,6,0,0
8,TAZ,0,15,0,0
9,WVZ,6,0,0,0


In [44]:
cams_per_da_aufbewahrung = imp_ueberwachungscams.groupby(['Verantwortliche Dienstabteilung', 'Aufbewahrungsdauer (Tage)'])[['Aussen digital','Innen digital']].sum()
cams_per_da_aufbewahrung.reset_index().sort_values('Verantwortliche Dienstabteilung', ascending=True)
#cams_per_da_aufbewahrung.reset_index().melt()

,Verantwortliche Dienstabteilung,Aufbewahrungsdauer (Tage),Aussen digital,Innen digital
0,EWZ,15,1,0
1,IMMO,7,852,0
2,IMMO,20,5,105
3,LSZ,7,0,33
4,MRZ,14,0,29
5,OIZ,17,1,0
6,STAPO,14,0,0
7,Stadtspital Triemli,14,274,25
8,Stadtspital Waid,14,11,6
9,TAZ,14,0,15


### Daten in interaktiver Treemap zeigen
Dazu gibt es eine sehr nützliche Webseite https://plotly.com/python/treemaps/
Zu Farbskalen, siehe: https://plotly.com/python/builtin-colorscales/

In [70]:
#imp_ueberwachungscams.dtypes

In [ ]:
#cams_per_da_aufbewahrung = imp_ueberwachungscams.groupby(['Verantwortliche Dienstabteilung', 'Aufbewahrungsdauer (Tage)'])[['Aussen digital','Innen digital']].sum()
df = cams_per_da_aufbewahrung.reset_index()


fig = px.treemap(df.reset_index(), path=[px.Constant("Test"), 'Verantwortliche Dienstabteilung', 'Aufbewahrungsdauer (Tage)', 'Aussen digital']
                 , values='Aussen digital'
                 , color='Aussen digital'
                 ## bei discrete scales (bei Klassen und Strings):                 
                 #, color_discrete_map={'(?)':'lightgrey','EVP':'gold', 'SP':'#FF3030', 'PdA':'#EE3B3B', 'Grüne':'#A2CD5A','GLP':'#CAFF70','Die Mitte':'orange','FDP':'#104E8B','AL':'deeppink','SVP':'forestgreen', 'FL ZÜRI':'#8B864E','Volt':'#lightblue' }                
                 ## bei continuous scales (bei Zahlenwerten):
                , color_continuous_scale='Blues'
                #, color_continuous_midpoint=np.average(df['AnzBestWir'], weights=df['AnzBestWir'])           
                )
fig.update_traces(root_color="grey")
fig.update_layout(margin = dict(t=50, l=25, r=250, b=25))
fig.show()

### Einfache Visualisierungen zur Plausi

Exploriere die Daten mit Pivottable.JS

In [71]:
from pivottablejs import pivot_ui

pivot_ui(imp_ueberwachungscams)

Liniendiagramm 
[Link zur Doku](https://altair-viz.github.io/gallery/multiline_highlight.html)

In [72]:
imp_ueberwachungscams.dtypes

Typ                                      object
Standort                                 object
Adresse                                  object
IMMO-Nr. (falls zutreffend)              object
Lat                                     float64
Long                                    float64
Aussen analog                           float64
Aussen digital                          float64
Innen analog                            float64
Innen digital                           float64
Portrait (Ein-bau in GSA)               float64
Undefiniert                             float64
Gebäude-zugang                           object
Schalter-bereich                         object
Liftvor-platz                            object
Fassade                                  object
Eingangsbereich                          object
Bereiche der Überwachung                 object
Zusätzliche Bereiche der Überwachung     object
Aufbewahrungsdauer (Tage)                 int64
Verantwortliche Dienstabteilung         

In [76]:
myAgg = imp_ueberwachungscams \
    .groupby(['Verantwortliche Dienstabteilung', 'Aufbewahrungsdauer (Tage)']) \
    .agg(sum_outside=('Aussen digital', 'sum'),sum_inside=('Innen digital', 'sum')) \
    .sort_values('sum_outside', ascending=False) 

myAgg.reset_index()


,Verantwortliche Dienstabteilung,Aufbewahrungsdauer (Tage),sum_outside,sum_inside
0,IMMO,7,852,0
1,Stadtspital Triemli,14,274,25
2,Stadtspital Waid,14,11,6
3,WVZ,10,6,0
4,IMMO,20,5,105
5,EWZ,15,1,0
6,OIZ,17,1,0
7,LSZ,7,0,33
8,MRZ,14,0,29
9,STAPO,14,0,0


In [78]:
# df_statzonen = df_statzonen[(df_statzonen.knr>0) & (df_statzonen.qname!="Hirslanden")]
#aktuellesJahr[[aktuellesJahr.KreisSort =7]])

highlight = alt.selection(type='single', on='mouseover',
                          fields=['Verantwortliche Dienstabteilung'], nearest=True)
#x='date:StichtagDatJahr',
base = alt.Chart(myAgg.reset_index().query('sum_outside>0')).encode(
    x='Verantwortliche Dienstabteilung',
    y='sum_outside',
    color='Aufbewahrungsdauer (Tage)'
)

points = base.mark_circle().encode(
    opacity=alt.value(0.75)
).add_selection(
    highlight
).properties(
    width=700 , height=500
)
lines = base.mark_line().encode(
    size=alt.condition(~highlight, alt.value(1.5), alt.value(4))
).interactive()

points + lines

alt.LayerChart(...)